# Bronze - State of Data Brasil (Tech Challenge)

Lê os CSVs da **Landing** (`landing/YYYY/`), padroniza os nomes das colunas, trata os dados e grava na **Bronze** em Parquet particionado por `ano_pesquisa`:

| Tabela (pasta em `bronze/`) | Conteúdo |
|---|---|
| `respondentes` | 1 linha por respondente, campos de perfil/trabalho com nomes iguais em todos os anos |
| `respostas_multipla` | formato longo das perguntas de múltipla escolha: `id_respondente, pergunta, opcao, selecionado` |
| `dim_perguntas` | dicionário: coluna original -> nome na Bronze (linhagem/metadados) |

Reprocessar um ano só substitui a partição daquele ano (`partitionOverwriteMode=dynamic`).

In [ ]:
%idle_timeout 30
%worker_type G.1X
%number_of_workers 2

In [ ]:
import re, unicodedata
from pyspark.sql import functions as F

BUCKET = "tech-challenge-datahackers"
BASE   = f"s3://{BUCKET}"
ANOS   = [2024, 2025]            # anos a processar

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

## 1. Funções de padronização

- Campos comuns viram `slug(rótulo)` (sem o código da pergunta), porque **o código muda entre os anos** (ex.: banco de dados é `4.g` em 2024 e `4.d` em 2025).
- Opções de múltipla escolha (colunas 0/1) viram linhas em `respostas_multipla`, identificadas por `pergunta` + `opcao`.
- `ALIAS_PERGUNTA` junta perguntas que mudaram de nome entre os anos.

In [ ]:
def slug(txt, maxlen=60):
    t = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode()
    t = re.sub(r"[^a-zA-Z0-9]+", "_", t).strip("_").lower()
    t = re.sub(r"_?em_20\d{2}$", "", t)          # remove o ano do nome (ex.: layoff_em_2025)
    return t[:maxlen].rstrip("_")

COD = re.compile(r"^(\d+)((?:\.[a-z0-9]+)*)[_ ](.*)$", re.S)

def parse_coluna(c):
    m = COD.match(c)
    if not m:
        return None, None, c
    return m.group(1) + m.group(2), m.group(1), m.group(3)    # codigo, secao, rotulo

# perguntas que mudaram de nome entre os anos (2025 -> nome canônico de 2024)
ALIAS_PERGUNTA = {
    "s4_linguagem_preferida": "s4_linguagem_de_programacao_dia_a_dia",
}

# campos que viram boolean / inteiro
BOOL_COLS = ["vive_no_brasil", "vive_no_estado_de_formacao", "atua_como_gestor",
             "satisfeito_atualmente", "possui_data_lake", "possui_data_warehouse"]
INT_COLS  = ["idade"]

# erros de digitação encontrados na pesquisa de 2025
CORRECOES = {
    "faixa_salarial": {"de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30.000/mês"},
    "numero_de_funcionarios": {"de 501 a 100": "de 501 a 1000"},
}

In [ ]:
def montar_esquema(landing):
    """Classifica cada coluna do CSV e define o nome final na Bronze."""
    tipos = dict(landing.dtypes)
    info = {}
    for c in landing.columns:
        codigo, secao, rotulo = parse_coluna(c)
        info[c] = {"codigo": codigo, "secao": secao, "rotulo": rotulo}

    por_codigo = {}
    for c, v in info.items():
        if v["codigo"]:
            assert v["codigo"] not in por_codigo, f"codigo repetido: {v['codigo']}"
            por_codigo[v["codigo"]] = c

    numericos = ("tinyint", "smallint", "int", "bigint", "float", "double")
    flags = [c for c, v in info.items()
             if v["codigo"] and re.match(r"^\d+\.[a-z]\.\d+$", v["codigo"]) and tipos[c] in numericos]

    pais_de_flags = set()
    for c in flags:
        p = por_codigo.get(info[c]["codigo"].rsplit(".", 1)[0])
        if p:
            pais_de_flags.add(p)

    esquema = []     # (coluna_original, novo_nome, tipo, pergunta, opcao, rotulo)
    usados = set()
    for i, c in enumerate(landing.columns):
        v = info[c]
        if c in flags:
            pcod = v["codigo"].rsplit(".", 1)[0]
            pai = por_codigo.get(pcod)
            rot_pai = info[pai]["rotulo"] if pai else pcod
            pergunta = f"s{v['secao']}_{slug(rot_pai, 50)}"
            pergunta = ALIAS_PERGUNTA.get(pergunta, pergunta)
            esquema.append((c, f"flag_{i}", "flag", pergunta, slug(v["rotulo"]), v["rotulo"]))
        elif c in pais_de_flags:
            continue                                  # lista em texto: redundante com os flags
        else:
            nome = "id_respondente" if slug(v["rotulo"]) == "token" else slug(v["rotulo"])
            if nome in usados:
                nome = f"{nome}_{v['codigo'].replace('.', '_')}"
            usados.add(nome)
            esquema.append((c, nome, "campo", None, None, v["rotulo"]))
    return esquema

In [ ]:
def processar_ano(ano):
    landing = (spark.read.option("header", True).option("sep", ",").option("quote", '"')
           .option("escape", '"').option("multiLine", True).option("encoding", "UTF-8")
           .option("inferSchema", True).csv(f"{BASE}/landing/{ano}/"))
    print(f"[{ano}] bronze: {landing.count()} linhas x {len(landing.columns)} colunas")

    esquema = montar_esquema(landing)
    sel = [F.col(f"`{c}`").alias(novo) for (c, novo, *_ ) in esquema]
    df = landing.select(*sel)

    flags  = [(novo, perg, op) for (_, novo, tipo, perg, op, _) in esquema if tipo == "flag"]
    campos = [novo for (_, novo, tipo, *_ ) in esquema if tipo == "campo"]

    # --- limpeza ---
    df = df.dropDuplicates(["id_respondente"]).withColumn("ano_pesquisa", F.lit(ano))
    df = df.withColumn("data_hora_envio", F.to_timestamp("data_hora_envio", "dd/MM/yyyy HH:mm:ss"))

    for col, mapa in CORRECOES.items():
        if col in df.columns:
            df = df.replace(mapa, subset=[col])

    for c in BOOL_COLS:
        if c in df.columns:
            s = F.col(c).cast("string")
            df = df.withColumn(c, F.when(s.isin("True", "true", "1", "1.0"), F.lit(True))
                                   .when(s.isin("False", "false", "0", "0.0"), F.lit(False)))
    for c in INT_COLS:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast("int"))

    # faixa salarial -> minimo, maximo, ponto medio
    if "faixa_salarial" in df.columns:
        limpo = F.regexp_replace("faixa_salarial", r"\.", "")
        primeiro = F.regexp_extract(limpo, r"(\d+)", 1)
        segundo  = F.regexp_extract(limpo, r"a R\$ (\d+)", 1)
        menos    = F.col("faixa_salarial").startswith("Menos")
        df = (df.withColumn("salario_min", F.when(menos, F.lit(0)).otherwise(primeiro.cast("int")))
                .withColumn("salario_max", F.when(menos, primeiro.cast("int"))
                                            .otherwise(F.when(segundo != "", segundo.cast("int"))))
                .withColumn("salario_medio", F.coalesce((F.col("salario_min") + F.col("salario_max")) / 2,
                                                        F.col("salario_min"))))

    # demais campos de texto: tipo string (evita conflito de tipos entre anos)
    tratados = set(BOOL_COLS + INT_COLS + ["data_hora_envio", "ano_pesquisa", "salario_min",
                                           "salario_max", "salario_medio"])
    for c in campos:
        if c not in tratados:
            df = df.withColumn(c, F.col(c).cast("string"))

    # --- tabela de respondentes e tabela longa ---
    resp = df.select("id_respondente", "ano_pesquisa",
                     *[c for c in df.columns if c in campos and c != "id_respondente"],
                     *[c for c in ["salario_min", "salario_max", "salario_medio"] if c in df.columns])

    estruturas = [F.struct(F.lit(p).alias("pergunta"), F.lit(o).alias("opcao"),
                           F.col(n).cast("int").alias("selecionado")) for (n, p, o) in flags]
    longo = (df.select("id_respondente", "ano_pesquisa", F.explode(F.array(*estruturas)).alias("t"))
               .select("id_respondente", "ano_pesquisa", "t.pergunta", "t.opcao", "t.selecionado")
               .where(F.col("selecionado").isNotNull()))

    # --- dicionario (linhagem) ---
    linhas = [(ano, c, (novo if tipo == "campo" else f"{perg}/{op}"), tipo, perg, op, rot)
              for (c, novo, tipo, perg, op, rot) in esquema]
    dim = spark.createDataFrame(
        linhas, "ano_pesquisa int, coluna_original string, coluna_bronze string, tipo string, "
                "pergunta string, opcao string, rotulo_original string")
    return resp, longo, dim

## 2. Executar e gravar na Bronze

In [ ]:
def gravar(df, nome):
    df.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(f"{BASE}/bronze/{nome}/")

for ano in ANOS:
    resp, longo, dim = processar_ano(ano)
    gravar(resp,  "respondentes")
    gravar(longo, "respostas_multipla")
    gravar(dim,   "dim_perguntas")
    print(f"[{ano}] bronze gravada")

## 3. Validação

Confira: nº de respondentes por ano (deve ser menor ou igual ao da Bronze, por causa das duplicatas) e a % de uso de cada linguagem,
calculada **só entre quem viu a pergunta** (as colunas 0/1 vazias significam "pergunta não exibida").

In [ ]:
resp  = spark.read.parquet(f"{BASE}/bronze/respondentes/")
longo = spark.read.parquet(f"{BASE}/bronze/respostas_multipla/")

resp.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()

(longo.where(F.col("pergunta") == "s4_linguagem_de_programacao_dia_a_dia")
      .groupBy("ano_pesquisa", "opcao")
      .agg(F.countDistinct("id_respondente").alias("nu_resp"),
           F.round(F.avg("selecionado") * 100, 1).alias("%_usa"))
      .where(F.col("opcao").isin("sql", "python", "r"))
      .orderBy("opcao", "ano_pesquisa").show())